We want to answer some questions based on the data below. Here is a breakdown of what we are investigating.
: Do yellow taxi passengers in New York City tip drivers more in the winter or summer?

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

taxi = pd.read_csv('taxi.csv')
taxi.describe()

In [ ]:
taxi.head()

In [ ]:
taxi.tail()

I change the data pickup times and only worked with the months so that I can have an accurate view of months of winter and summer.

In [ ]:
taxi ['pick_up_month'] = pd.to_datetime(taxi['tpep_pickup_datetime']).dt.month
taxi ['drop_off_month'] = pd.to_datetime(taxi['tpep_dropoff_datetime']).dt.month

In [ ]:
print(taxi['pick_up_month'].tail())

In [ ]:
print(taxi.columns.tolist())

I plotted pick_up_month against tips to see the relationship and visualized it and we can see that there is a small upward trend concluding that passengers often tip more during summertimes.

In [ ]:
sns.relplot(x='pick_up_month', y='tip_amount', data=taxi, kind='line')

Back with more investigations. Here we will only focus on credit payments.

In [ ]:
taxi['payment_type'].unique()

In [ ]:
print("Converting datetime columns in the 'taxi' DataFrame...")
taxi['tpep_pickup_datetime'] = pd.to_datetime(taxi['tpep_pickup_datetime'], errors='coerce')
taxi['tpep_dropoff_datetime'] = pd.to_datetime(taxi['tpep_dropoff_datetime'], errors='coerce')

In [ ]:
#filter for credit card payments
taxi_credit_card = taxi[taxi['payment_type'] == 1].copy()
print(f"\nOriginal DataFrame size: {len(taxi)} trips")
print(f"Credit Card trips only: {len(taxi_credit_card)} trips")
print(f"Percentage of Credit Card trips: {len(taxi_credit_card) / len(taxi) * 100:.2f}%")

Feature Engineering: Tip Percentage and Time-based Features
Calculating tip percentage provides a normalized view of tipping behavior, independent of the absolute fare amount. Extracting month, hour, and day of week allows for granular temporal analysis.

In [ ]:
# --- 1. Calculate Tip Percentage ---
# Handle cases where fare_amount might be 0 to avoid division by zero
# If fare_amount is 0, tip_percentage should also be 0 or NaN, or exclude such trips.
# We'll set it to 0 if fare is 0.
taxi_credit_card['tip_percentage'] = (taxi_credit_card['tip_amount'] / taxi_credit_card['fare_amount']) * 100
taxi_credit_card.loc[taxi_credit_card['fare_amount'] == 0, 'tip_percentage'] = 0

# Handle potential infinite values if fare_amount was extremely small but not zero (unlikely but good practice)
taxi_credit_card.replace([np.inf, -np.inf], np.nan, inplace=True)
# Fill any resulting NaNs (e.g., from small fare_amount + tip) with 0 or drop them
taxi_credit_card['tip_percentage'].fillna(0, inplace=True)

# Cap extreme outliers for tip percentage (e.g., tips > 100% of fare are usually data errors)
# You can decide on a reasonable cap based on your df.describe() for 'tip_percentage'
# Let's cap at 50% for demonstration, typical tips are 15-20%

taxi_credit_card['tip_percentage'] = taxi_credit_card['tip_percentage'].clip(upper=50)

print("\n Tip percentage calculated and outliers capped" )
print(taxi_credit_card[['fare_amount', 'tip_amount', 'tip_percentage']].describe())

# --- 2. Extract Temporal Features ---
# Now, these lines should work without error:
taxi_credit_card['pickup_month'] = taxi_credit_card['tpep_pickup_datetime'].dt.month # Changed to pickup_datetime
taxi_credit_card['pickup_hour'] = taxi_credit_card['tpep_pickup_datetime'].dt.hour   # Changed to pickup_datetime
taxi_credit_card['pickup_day_week'] = taxi_credit_card['tpep_pickup_datetime'].dt.dayofweek # Changed to pickup_datetime

# --- 3. Define Seasons ---
# Winter: Dec (12), Jan (1), Feb (2)
# Summer: Jun (6), Jul (7), Aug (8)

def get_season(month):
    if month in [12,1,2]:
        return 'Winter'
    elif month in [6,7,8]:
        return 'Summer'
    elif month in [3,4,5]:
        return 'spring'
    elif month in [9,10,11]:
        return 'Autumn'
    return 'Unknown'

taxi_credit_card['season'] = taxi_credit_card['pick_up_month'].apply(get_season)
print("\nTemporal features and 'season' column created.")
print(taxi_credit_card['season'].value_counts())
    


In [ ]:
seasonal_tips = taxi_credit_card.groupby('season')['tip_percentage'].mean().reindex(['Winter','Summer','Autumn','spring'])
print("\nAverage tip percentage by season: ")
print(seasonal_tips)

In [ ]:
# Plotting Seasonal Tip Percentage
plt.figure(figsize=(10, 6))
seasonal_tips.plot(kind='bar', color=sns.color_palette("pastel"))
plt.title('Average Tip Percentage by Season (Credit Card Trips)')
plt.xlabel('Season')
plt.ylabel('Average Tip Percentage (%)')
plt.xticks(rotation=45)
plt.show()

We can see that tips are higher during the Summer.

In [ ]:
# --- 2. Influence of Time of Day ---
hourly_tips = taxi_credit_card.groupby('pickup_hour')['tip_percentage'].mean()
print("\nAverage tip percentage by hour of the day: ")
print(hourly_tips)

In [ ]:
plt.figure(figsize=(12, 6))
hourly_tips.plot(kind='line', marker='o', color='purple')
plt.title('Average Tip Percentage by Hour of Day (Credit Card Trips)')
plt.xlabel('Hour of Day')
plt.ylabel('Average Tip Percentage (%)')
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()

Her we can see Average Tip Percentage by Hour of day .

In [ ]:
# --- 3. Influence of Day of Week ---
# Map dayofweek numbers to names for better readability
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
taxi_credit_card['pickup_day_name'] = taxi_credit_card['pickup_day_week'].map(day_names)

daily_tips = taxi_credit_card.groupby('pickup_day_name')['tip_percentage'].mean().reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
print("\nAverage Tip Percentage by Day of Week (Credit Card Trips Only):")
print(daily_tips)

plt.figure(figsize=(10, 6))
daily_tips.plot(kind='bar', color=sns.color_palette("cividis"))
plt.title('Average Tip Percentage by Day of Week (Credit Card Trips)')
plt.xlabel('Day of Week')
plt.ylabel('Average Tip Percentage (%)')
plt.xticks(rotation=45)
plt.show()

Average Tip Percentage by Day of Week.

In [ ]:
# --- 4. Influence of Trip Distance / Fare Amount ---
# Scatter plot for tip percentage vs. trip distance
plt.figure(figsize=(12, 7))
sns.scatterplot(x='trip_distance', y='tip_percentage', data=taxi_credit_card.sample(n=min(10000, len(taxi_credit_card))), alpha=0.1, color='blue') # Sample for performance
plt.title('Tip Percentage vs. Trip Distance (Credit Card Trips)')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Tip Percentage (%)')
plt.xlim(0, 30) # Limit x-axis to common trip distances
plt.ylim(0, 30) # Limit y-axis to common tip percentages
plt.show()

# Relationship between Fare Amount and Tip Amount (as a percentage of Total Amount)
# Useful to see if higher total amounts lead to lower/higher percentage tips
plt.figure(figsize=(12, 7))
sns.scatterplot(x='fare_amount', y='tip_percentage', data=taxi_credit_card.sample(n=min(10000, len(taxi_credit_card))), alpha=0.1, color='blue')
plt.title('Tip Percentage vs. Fare Amount (Credit Card Trips)')
plt.xlabel('Fare Amount ($)')
plt.ylabel('Tip Percentage (%)')
plt.xlim(0, 100)
plt.ylim(0, 30)
plt.show()

In [ ]:
# --- 5. Influence of Passenger Count ---
passenger_tips = taxi_credit_card.groupby('passenger_count')['tip_percentage'].mean().sort_index()
print("\nAverage Tip Percentage by Passenger Count (Credit Card Trips Only):")
print(passenger_tips)

plt.figure(figsize=(10, 6))
passenger_tips.plot(kind='bar', color=sns.color_palette("plasma"))
plt.title('Average Tip Percentage by Passenger Count (Credit Card Trips)')
plt.xlabel('Passenger Count')
plt.ylabel('Average Tip Percentage (%)')
plt.xticks(rotation=0)
plt.show()
